In [1]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    pipeline
)

# ------- PARAMETROS y ARCHIVOS -------
RUTA_CATEGORIAS = "categorias.txt"
RUTA_RECLAMOS = "textos_reclamos_resp_empresa.xlsx"
COLS = [
    "Tema - Peticion Concreta",
    "Repuesta de Empresa (1era)",
    "Clasificacion que le asignó el Analista "
]

# 1. Cargar y limpiar categorías
with open(RUTA_CATEGORIAS, 'r', encoding='utf-8') as f:
    CATEGORIAS = [line.strip() for line in f if line.strip()]

# Mapeos para el entrenamiento
label2id = {label: i for i, label in enumerate(CATEGORIAS)}
id2label = {i: label for i, label in enumerate(CATEGORIAS)}

# 2. Cargar datos históricos para validación y entrenamiento [cite: 25]
df_reclamos = pd.read_excel(RUTA_RECLAMOS, skiprows=1, usecols=COLS).dropna()

class ReclamosDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

class ClasificadorReclamosSEC:
    def __init__(self, categorias, modelo_path=None):
        self.categorias = categorias
        # Usamos BETO (BERT en español) como base para el sector eléctrico [cite: 19]
        self.base_model = "dccuchile/bert-base-spanish-wwm-uncased"
        self.tokenizer = AutoTokenizer.from_pretrained(self.base_model)
        
        if modelo_path and os.path.exists(modelo_path):
            self.model = AutoModelForSequenceClassification.from_pretrained(modelo_path)
        else:
            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.base_model, 
                num_labels=len(categorias),
                id2label=id2label,
                label2id=label2id
            )

    def entrenar(self, df):
        print("Preparando entrenamiento masivo...")
        
        # Combinar columnas para dar contexto al asistente [cite: 10]
        textos = (df["Tema - Peticion Concreta"] + " [SEP] " + df["Repuesta de Empresa (1era)"]).tolist()
        labels = df["Clasificacion que le asignó el Analista "].str.strip().map(label2id).tolist()

        # Tokenización
        encodings = self.tokenizer(textos, truncation=True, padding=True, max_length=512)
        dataset = ReclamosDataset(encodings, labels)

        # Configuración del entrenamiento
        training_args = TrainingArguments(
            output_dir='./resultados_sec',
            num_train_epochs=3,
            per_device_train_batch_size=8,
            logging_steps=10,
            save_strategy="epoch",
            learning_rate=2e-5
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=dataset
        )

        print("Iniciando ajuste fino (Fine-tuning)...")
        trainer.train()
        self.model.save_pretrained("./modelo_final_sec")
        print("Modelo entrenado y guardado.")

    def clasificar(self, peticion, respuesta):
        """Inferencia utilizando el modelo especializado."""
        texto = f"{peticion} [SEP] {respuesta}"
        inputs = self.tokenizer(texto, return_tensors="pt", truncation=True, padding=True)
        
        with torch.no_grad():
            logits = self.model(**inputs).logits
        
        probs = torch.nn.functional.softmax(logits, dim=-1)
        score, pred_id = torch.max(probs, dim=-1)
        
        return {
            "categoria_sugerida": id2label[pred_id.item()],
            "confianza": score.item()
        }

# --- EJECUCIÓN DEL FLUJO REDISEÑADO [cite: 15] ---

asistente = ClasificadorReclamosSEC(CATEGORIAS)

fila = df_reclamos.iloc[3]
prediccion = asistente.clasificar(
    fila["Tema - Peticion Concreta"], 
    fila["Repuesta de Empresa (1era)"]
)

print("\n--- RESULTADO POST-ENTRENAMIENTO ---")
print(f"Analista Humano: {fila['Clasificacion que le asignó el Analista ']}")
print(f"Asistente IA:    {prediccion['categoria_sugerida']}")
print(f"Confianza:       {prediccion['confianza']:.2%}")

c:\Users\bnjca\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 18619.09it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias   


--- RESULTADO POST-ENTRENAMIENTO ---
Analista Humano: CONSUMO NO REGISTRADO
Asistente IA:    CONSUMO NO REGISTRADO
Confianza:       18.92%
